# Tuto how to train a GNN

## First method: Inductive Training

1. Import stuff

In [1]:
import os
import json
import numpy as np
import torch
from torch_geometric.data import DataLoader
from torch.utils.data import random_split
import torch.nn as nn
import torch_geometric.nn as pyg_nn

from models.dataset_inductive import GraphDatasetInductive
from models.model_utils import train_inductive, test_inductive

2. import your model (that you defined in the "/models" file)

In [2]:
from models.base_gat_combined import GATClassifierCombined


3. Define your parameters

In [3]:
# Paramaeters
json_dir          = './json_output/'
# Model parameters
in_channels       = 1280
hidden_channels   = 256
out_channels      = 256
num_layers        = 5
dropout           = 0.1
act               = 'relu'
# Training Prameters
lr                = 1e-2
weight_decay      = 5e-4
batch_size        = 2
epochs            = 10

device            = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

4. Load data and split it into training/Test dataset

In [6]:
# dataset + train/test split
full_dataset = GraphDatasetInductive(json_dir, struct_thresh=0.6, textural_thresh=0.4)
n_train      = int(0.8 * len(full_dataset))
n_test       = len(full_dataset) - n_train
train_ds, test_ds = random_split(full_dataset, [n_train, n_test])

train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
test_loader  = DataLoader(test_ds,  batch_size=batch_size)

5. Initialize your model

In [13]:
# model, optimizer, loss
model     = GATClassifierCombined(in_channels=in_channels, 
                            hidden_channels=hidden_channels, 
                            out_channels=out_channels, 
                            num_layers=num_layers, 
                            dropout=dropout, 
                            act=act).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
criterion = nn.BCELoss()

6. Train your model

In [ ]:
# training loop
for epoch in range(1, epochs+1):
    train_loss = train_inductive(model, train_loader, optimizer, criterion, device=device)
    train_n_acc, train_e_acc = test_inductive(model, train_loader, device=device)
    test_n_acc,  test_e_acc  = test_inductive(model, test_loader,  device=device)

    print(f'Epoch {epoch:02d} | '
          f'Loss: {train_loss:.4f} | '
          f'Train Node Acc: {train_n_acc:.4f} | Train Edge Acc: {train_e_acc:.4f} | '
          f'Test Node Acc:  {test_n_acc:.4f} | Test Edge Acc:  {test_e_acc:.4f}')

Exception ignored in: <bound method IPythonKernel._clean_thread_parent_frames of <ipykernel.ipkernel.IPythonKernel object at 0x106629910>>
Traceback (most recent call last):
  File "/Users/marvin/Library/Python/3.11/lib/python/site-packages/ipykernel/ipkernel.py", line 775, in _clean_thread_parent_frames
    def _clean_thread_parent_frames(

KeyboardInterrupt: 


## Second method: Transductive Training

1. Import stuff

In [7]:
import os
import json
import numpy as np
import torch
from torch_geometric.data import DataLoader
from torch.utils.data import random_split
import torch.nn as nn
import torch_geometric.nn as pyg_nn

from models.dataset_transductive import GraphDatasetTransductive # This changed
from models.model_utils import train_combined, test_combined # This changed

2. import your model (that you defined in the "/models" file)

In [8]:
from models.base_gat_combined import GATClassifierCombined


3. Define your parameters

In [9]:
# Paramaeters
json_dir          = './json_output/'
# Model parameters
in_channels       = 1280
hidden_channels   = 256
out_channels      = 256
num_layers        = 5
dropout           = 0.1
act               = 'relu'
# Training Prameters
lr                = 1e-2
weight_decay      = 5e-4
batch_size        = 2
epochs            = 10

device            = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

4. Load data and split it into training/Test dataset

In [10]:
# Initialise dataset and dataloader
dataset = GraphDatasetTransductive(json_dir, struct_thresh=0.6, textural_thresh=0.4)  # This changed

loader = DataLoader(dataset, batch_size=batch_size, shuffle=True) # This changed

5. Initialize your model

In [11]:
# Initialize model
model = GATClassifierCombined(in_channels=in_channels, 
                            hidden_channels=hidden_channels, 
                            out_channels=out_channels, 
                            num_layers=num_layers, 
                            dropout=dropout, 
                            act=act).to(device)

# Loss function and Optimizer
optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
criterion = torch.nn.BCELoss()

6. Train your model

In [12]:
# training loop    # This changed
for epoch in range(1, epochs+1):
    loss = train_combined(model, optimizer, criterion, loader, node_loss_weight=1.0, edge_loss_weight=1.0)
    train_acc_struct, test_acc_struct, train_acc_coplanarity, test_acc_coplanarity = test_combined(model, criterion, loader,
                                                                                                   threshold_structural=0.5,
                                                                                                   threshold_coplanarity=0.5)
    
    print(f'Epoch: {epoch:03d}, Loss: {loss:.4f}')
    print(f'Structural/Textural: Train Acc: {train_acc_struct:.4f}, Test Acc: {test_acc_struct:.4f}')
    print(f'Coplanarity: Train Acc: {train_acc_coplanarity:.4f}, Test Acc: {test_acc_coplanarity:.4f}')

Epoch: 001, Loss: 1.3641
Structural/Textural: Train Acc: 0.6834, Test Acc: 0.6833
Coplanarity: Train Acc: 0.9399, Test Acc: 0.9431
Epoch: 002, Loss: 35.6324
Structural/Textural: Train Acc: 0.6741, Test Acc: 0.7056
Coplanarity: Train Acc: 0.9319, Test Acc: 0.9519
Epoch: 003, Loss: 39.2366
Structural/Textural: Train Acc: 0.7058, Test Acc: 0.6889
Coplanarity: Train Acc: 0.9338, Test Acc: 0.9399
Epoch: 004, Loss: 37.1718
Structural/Textural: Train Acc: 0.6704, Test Acc: 0.6833
Coplanarity: Train Acc: 0.9415, Test Acc: 0.9353
Epoch: 005, Loss: 37.3744
Structural/Textural: Train Acc: 0.6890, Test Acc: 0.6833
Coplanarity: Train Acc: 0.9364, Test Acc: 0.9385
Epoch: 006, Loss: 36.6295
Structural/Textural: Train Acc: 0.7002, Test Acc: 0.6389
Coplanarity: Train Acc: 0.9415, Test Acc: 0.9339
Epoch: 007, Loss: 37.1882
Structural/Textural: Train Acc: 0.6983, Test Acc: 0.6611
Coplanarity: Train Acc: 0.9418, Test Acc: 0.9439
Epoch: 008, Loss: 35.7799
Structural/Textural: Train Acc: 0.6797, Test Acc: 0